# DentaMind AI - grouped 5-fold cross-validation of the BASE finding classifier (Kaggle GPU)

Pre-registered protocol: `reports/cv5_protocol.md` (2026-10-24). Approved fold file `data/splits/cv5_folds_v1.csv` (P4 option B).
Unit = radiograph (DENTEX has no patient ids). BASE recipe unchanged; 5 folds x seeds 42-46 = 25 runs. Epoch selection on a
group-disjoint inner validation subset of the training folds; every held-out fold is predicted once per seed. **No test or
calibration image is read.** No new model is compared and no improvement is claimed.

**Before running:** GPU T4, Internet **On** (DENTEX images from the official archive), Add Input -> the private bundle dataset
(`dentamind_kaggle_bundle`, built with `--with-deployed-checkpoint --with-external`). Runs are resumable: if the session stops, add
this notebook's previous output as input and set `RESUME_FROM`. Output: `dentamind_5fold_cv_artifacts.zip`.

In [ ]:
# ---- configuration -----------------------------------------------------------------------------------------------
import os, sys, json, time, shutil, subprocess, glob, zipfile, hashlib, dataclasses, collections
from pathlib import Path
import numpy as np

SEEDS = [42, 43, 44, 45, 46]          # protocol section 3
FOLDS = [0, 1, 2, 3, 4]               # all five folds; never reduced
TIME_BUDGET_H = 11.0                  # Kaggle sessions stop at 12 h; no new run starts after this budget
RESUME_FROM = None                    # e.g. "/kaggle/input/<this notebook's previous output>/cv5/runs"

# Local pipeline check only (never on Kaggle): DENTAMIND_SMOKE=1 -> 1 epoch, seed 42, all 5 folds; output marked SMOKE.
SMOKE = os.environ.get("DENTAMIND_SMOKE") == "1"
if SMOKE:
    SEEDS = [42]

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path(os.environ.get("DENTAMIND_WORK") or ("/kaggle/working" if ON_KAGGLE else "kaggle_work_cv5")).resolve()
ART = WORK / "cv5"
RUNS = ART / "runs"
DATA_ROOT = Path(os.environ.get("DENTAMIND_DATA_ROOT") or ("/tmp/dentamind_data" if ON_KAGGLE else WORK / "data"))
CODE = WORK / "code"
CACHE = Path("/tmp/dentamind_cache_cv5") if ON_KAGGLE else WORK / "cache"
for d in (ART, RUNS, DATA_ROOT, CODE, CACHE):
    d.mkdir(parents=True, exist_ok=True)
T0 = time.time()
LOG = ART / "training_log.txt"

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    with open(LOG, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%Y-%m-%d %H:%M:%S ") + s + "\n")

if RESUME_FROM and Path(RESUME_FROM).is_dir():
    shutil.copytree(RESUME_FROM, RUNS, dirs_exist_ok=True)
    log("resumed run folders from", RESUME_FROM)
log("CV5", "SMOKE (pipeline check, not a result)" if SMOKE else "full run", "seeds", SEEDS, "folds", FOLDS)

In [ ]:
# ---- 1. GPU and software ----------------------------------------------------------------------------------------------
import numpy as np, PIL, platform, torch, torchvision
env = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__, "numpy": np.__version__,
       "pillow": PIL.__version__, "cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "gpu_count": torch.cuda.device_count(),
       "disk_free_gb_working": round(shutil.disk_usage(WORK).free / 1e9, 1), "disk_free_gb_data": round(shutil.disk_usage(DATA_ROOT).free / 1e9, 1),
       "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
try:
    env["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=20).stdout.strip()
except Exception as e:
    env["nvidia_smi"] = f"unavailable: {e}"
log(json.dumps(env, indent=1))
(ART / "environment.json").write_text(json.dumps(env, indent=1))
if not torch.cuda.is_available():
    log("WARNING: no GPU. Enable Accelerator = GPU; 35+ ResNet-50 runs on CPU are not practical.")

In [ ]:
# ---- 2. locate the code/manifest bundle -------------------------------------------------------------------------------
def find_bundle():
    for z in ([os.environ["DENTAMIND_BUNDLE"]] if os.environ.get("DENTAMIND_BUNDLE") else []) +              glob.glob("/kaggle/input/**/dentamind_kaggle_bundle.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(CODE)
        return f"zip {z}"
    for p in glob.glob("/kaggle/input/**/kaggle/dentamind_kaggle.py", recursive=True):   # Kaggle auto-extracted the zip
        shutil.copytree(Path(p).parent.parent, CODE, dirs_exist_ok=True)
        return f"folder {Path(p).parent.parent}"
    repo = Path.cwd().parent if (Path.cwd().parent / "kaggle/dentamind_kaggle.py").is_file() else None   # local run from kaggle/
    if repo:
        shutil.copytree(repo / "kaggle", CODE / "kaggle", dirs_exist_ok=True)
        for sub in ("scripts/dataset", "data/splits", "reports"):
            shutil.copytree(repo / sub, CODE / sub, dirs_exist_ok=True)
        for f in ("data/raw/external/dentex_hf/validation_triple.json", "docs/audit_evidence/ml_improvement/crop_robustness/val_detector_boxes.json",
                  "docs/audit_evidence/ml_improvement/imbalance/results.json"):
            (CODE / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(repo / f, CODE / f)
        return f"local repository {repo}"
    raise SystemExit("bundle not found: attach the private dataset that contains dentamind_kaggle_bundle.zip")

log("bundle:", find_bundle())
bm = CODE / "BUNDLE_MANIFEST.json"
if bm.is_file():
    want = json.loads(bm.read_text())["files"]
    bad = [f for f, h in want.items() if hashlib.sha256((CODE / f).read_bytes()).hexdigest() != h]
    assert not bad, f"bundle files modified or corrupted: {bad}"
    log(f"bundle verified: {len(want)} files, SHA-256 match")
for p in (CODE / "kaggle", CODE / "scripts/dataset"):
    sys.path.insert(0, str(p))
import dentamind_kaggle as DK
import download_external as DX
if SMOKE:
    for c in DK.EXPERIMENTS.values():
        c.epochs = 1
from dsutil import CLASSES, sha256_file

In [ ]:
# ---- 2b. files not in the bundle (embedded, hash-checked) -----------------------------------------------------------
EMBEDDED = {"kaggle/cv5_lib.py": "\"\"\"Grouped 5-fold cross-validation of the BASE finding classifier. Protocol: reports/cv5_protocol.md (pre-registered 2026-10-24).\n\nReuses kaggle/dentamind_kaggle.py (crops, BASE recipe, isolated training, prediction, metrics) and scripts/dataset/make_cv_folds.py\n(group assignment). This module only adds: fold/leakage gates, the per-fold plan (fit / inner-validation / held-out), a detector that\nreproduces the production inference path, out-of-fold aggregation, a radiograph-level bootstrap and the Caries vs Deep Caries analysis.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport dataclasses\nimport hashlib\nimport json\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\nfor p in (HERE, HERE.parent / \"scripts\" / \"dataset\"):\n    if p.is_dir() and str(p) not in sys.path:\n        sys.path.insert(0, str(p))\nimport dentamind_kaggle as DK  # noqa: E402\nimport make_cv_folds as MCF  # noqa: E402\nfrom dsutil import CLASSES  # noqa: E402\n\nFOLD_FILE_SHA256 = \"9b767a4822844ccf0c4bf33d299e6e37420d1458a38035af0c3a3cb9107b47a6\"   # protocol section 2\nK = 5\nSEEDS = [42, 43, 44, 45, 46]\nINNER_K = 8\nPRIMARY = \"dentex_train_disease\"\nSECONDARY = \"dentex_official_val\"\nPOOL_SPLITS = (\"train\", \"validation\")\nEXCLUDED_SPLITS = (\"calibration\", \"test\")\nBOOT_N, BOOT_SEED = 2000, 0\nDET_IOU_MEDIAN_MIN = 0.98\n\n\nclass CVError(RuntimeError):\n    pass\n\n\ndef sha256_file(p) -> str:\n    h = hashlib.sha256()\n    with open(p, \"rb\") as fh:\n        for b in iter(lambda: fh.read(1 << 20), b\"\"):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef run_name(fold: int) -> str:\n    return f\"CV5_BASE_f{fold}\"\n\n\ndef fold_config(fold: int) -> DK.ExpConfig:\n    \"\"\"The BASE recipe, unchanged except for the run name (protocol section 3).\"\"\"\n    return dataclasses.replace(DK.EXPERIMENTS[\"BASE\"], name=run_name(fold),\n                               note=f\"CV5 fold {fold}: BASE recipe (C0r), fit on 4 folds minus inner validation; reports/cv5_protocol.md\")\n\n\n# ------------------------------------------------------------------------------------------------------------- folds (gate 1)\ndef load_folds(path, expected_sha=FOLD_FILE_SHA256) -> dict:\n    \"\"\"{radiograph_group: fold} from the approved fold file; refuses any other file.\"\"\"\n    got = sha256_file(path)\n    if expected_sha and got != expected_sha:\n        raise CVError(f\"fold file sha256 {got[:16]}... is not the approved {expected_sha[:16]}...\")\n    fold_of = {}\n    with open(path, encoding=\"utf-8\") as fh:\n        for r in csv.DictReader(fh):\n            g, f = r[\"radiograph_group\"], int(r[\"fold\"])\n            if g in fold_of:\n                raise CVError(f\"group {g} listed twice in the fold file\")\n            if not 0 <= f < K:\n                raise CVError(f\"group {g}: fold {f} outside 0..{K - 1}\")\n            fold_of[g] = f\n    return fold_of\n\n\ndef check_folds(fold_of: dict, rows_by_split: dict) -> dict:\n    \"\"\"Gate 1: every pool group exactly once; no calibration/test group in the pool; no image (SHA-256) in two folds.\"\"\"\n    pool = [r for s in POOL_SPLITS for r in rows_by_split[s]]\n    pool_groups = {r[\"radiograph_group\"] for r in pool}\n    excluded = {r[\"radiograph_group\"] for s in EXCLUDED_SPLITS for r in rows_by_split.get(s, [])}\n    excluded_sha = {r[\"image_sha256\"] for s in EXCLUDED_SPLITS for r in rows_by_split.get(s, [])}\n    problems = []\n    if set(fold_of) != pool_groups:\n        problems.append(f\"fold file groups != pool groups (missing {len(pool_groups - set(fold_of))}, extra {len(set(fold_of) - pool_groups)})\")\n    if set(fold_of) & excluded:\n        problems.append(f\"{len(set(fold_of) & excluded)} calibration/test groups in the fold file\")\n    sha_folds = defaultdict(set)\n    for r in pool:\n        sha_folds[r[\"image_sha256\"]].add(fold_of.get(r[\"radiograph_group\"]))\n        if r[\"image_sha256\"] in excluded_sha:\n            problems.append(f\"pool image {r['image_path']} is byte-identical to a calibration/test image\")\n    multi = [s for s, f in sha_folds.items() if len(f) > 1]\n    if multi:\n        problems.append(f\"{len(multi)} images in two folds\")\n    if problems:\n        raise CVError(\"fold gate failed: \" + \"; \".join(problems[:5]))\n    per_fold = defaultdict(Counter)\n    for r in pool:\n        per_fold[fold_of[r[\"radiograph_group\"]]][(r[\"source_dataset\"], r[\"target_class\"])] += 1\n    return {\"pool_groups\": len(pool_groups), \"pool_findings\": len(pool), \"excluded_groups_checked\": len(excluded),\n            \"radiographs_per_fold\": dict(sorted(Counter(fold_of.values()).items())),\n            \"findings_per_fold\": {f: {f\"{s}:{c}\": n for (s, c), n in sorted(cnt.items())} for f, cnt in sorted(per_fold.items())}}\n\n\n# ------------------------------------------------------------------------------------------------------------- plan (gate 2)\ndef fold_plan(pool_rows: list, fold_of: dict, fold: int) -> dict:\n    \"\"\"Protocol section 3. Returns row lists: fit / inner (BASE data rule: primary source only), heldout (primary), heldout_secondary.\"\"\"\n    heldout = [r for r in pool_rows if fold_of[r[\"radiograph_group\"]] == fold]\n    train_side = [r for r in pool_rows if fold_of[r[\"radiograph_group\"]] != fold and r[\"source_dataset\"] == PRIMARY]\n    inner_of = MCF.assign_folds(train_side, k=INNER_K, seed=42 + fold, with_impacted=True)\n    inner = [r for r in train_side if inner_of[r[\"radiograph_group\"]] == 0]\n    fit = [r for r in train_side if inner_of[r[\"radiograph_group\"]] != 0]\n    return {\"fold\": fold, \"fit\": fit, \"inner\": inner,\n            \"heldout\": [r for r in heldout if r[\"source_dataset\"] == PRIMARY],\n            \"heldout_secondary\": [r for r in heldout if r[\"source_dataset\"] == SECONDARY]}\n\n\ndef check_plan(plan: dict, fold_of: dict) -> dict:\n    \"\"\"Gate 2: fit, inner and held-out are pairwise disjoint by group and by image; held-out = exactly fold k.\"\"\"\n    k = plan[\"fold\"]\n    parts = {n: plan[n] for n in (\"fit\", \"inner\", \"heldout\", \"heldout_secondary\")}\n    g = {n: {r[\"radiograph_group\"] for r in rs} for n, rs in parts.items()}\n    s = {n: {r[\"image_sha256\"] for r in rs} for n, rs in parts.items()}\n    names = list(parts)\n    for i, a in enumerate(names):\n        for b in names[i + 1:]:\n            if g[a] & g[b] or s[a] & s[b]:\n                raise CVError(f\"fold {k}: {a} and {b} share {len(g[a] & g[b])} groups / {len(s[a] & s[b])} images\")\n    if any(fold_of[x] == k for x in g[\"fit\"] | g[\"inner\"]):\n        raise CVError(f\"fold {k}: a held-out radiograph is on the training side\")\n    if any(fold_of[x] != k for x in g[\"heldout\"] | g[\"heldout_secondary\"]):\n        raise CVError(f\"fold {k}: held-out set contains another fold\")\n    if {r[\"source_dataset\"] for r in plan[\"fit\"] + plan[\"inner\"]} - {PRIMARY}:\n        raise CVError(f\"fold {k}: non-BASE source in the fitting data\")\n    if not plan[\"fit\"] or not plan[\"inner\"] or not plan[\"heldout\"]:\n        raise CVError(f\"fold {k}: an empty part\")\n    return {n: {\"radiographs\": len(g[n]), \"findings\": len(parts[n]),\n                \"classes\": dict(sorted(Counter(r[\"target_class\"] for r in parts[n]).items()))} for n in names}\n\n\n# ------------------------------------------------------------------------------------------------------------- detector\ndef load_gray01(path) -> np.ndarray:\n    \"\"\"Same as dentexmodel.data.preprocess.load_gray (production input of the detector).\"\"\"\n    from PIL import Image\n    with Image.open(path) as im:\n        im.load()\n        if im.mode in (\"I;16\", \"I;16L\", \"I;16B\", \"I\"):\n            arr = np.asarray(im, dtype=np.float32)\n            top = float(arr.max()) or 1.0\n            return np.clip(arr / (65535.0 if top > 255 else 255.0), 0, 1).astype(np.float32)\n        if im.mode in (\"RGBA\", \"LA\", \"P\"):\n            im = im.convert(\"RGBA\").convert(\"L\") if im.mode != \"LA\" else im.convert(\"L\")\n        else:\n            im = im.convert(\"L\")\n        return np.asarray(im, dtype=np.float32) / 255.0\n\n\nclass ProductionDetector:\n    \"\"\"dentexmodel.inference.torch_models._run_detector for dentex_teeth_detection_v1: grayscale replicated to 3 channels, float [0,1],\n    autocast on CUDA (inference.mixed_precision default True), score >= DK.DET_SCORE (0.3).\"\"\"\n\n    def __init__(self, ckpt, device=None):\n        import torch\n        import torchvision as tv\n        self.torch = torch\n        self.device = device or torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        m = tv.models.detection.fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=2)\n        m.load_state_dict(torch.load(ckpt, map_location=\"cpu\", weights_only=True), strict=True)\n        self.model = m.to(self.device).eval()\n\n    def boxes(self, path) -> list:\n        t = self.torch\n        g = load_gray01(path)\n        rgb01 = np.repeat(g[..., None], 3, axis=2)\n        x = t.from_numpy(np.ascontiguousarray(rgb01.transpose(2, 0, 1))).float().to(self.device)\n        with t.no_grad(), t.autocast(device_type=self.device.type, enabled=self.device.type == \"cuda\"):\n            o = self.model([x])[0]\n        return [[float(v) for v in b] + [float(s)] for b, s in zip(o[\"boxes\"].float().cpu().tolist(), o[\"scores\"].float().cpu().tolist())\n                if s >= DK.DET_SCORE]\n\n\ndef gt_boxes(rows) -> list:\n    out = []\n    for r in rows:\n        x, y, w, h = json.loads(r[\"bbox_xywh\"])\n        out.append([x, y, x + w, y + h])\n    return out\n\n\ndef compare_detector(rows, new: dict, cached: dict, key=lambda r: r[\"project_image_id\"]) -> dict:\n    \"\"\"Gate 3 statistic: per image, match GT findings to cached and to regenerated boxes (DK.match); the matched finding sets must be\n    identical, and corresponding boxes must agree (median IoU).\"\"\"\n    by_img = defaultdict(list)\n    for r in rows:\n        by_img[key(r)].append(r)\n    same, diff, ious = 0, [], []\n    for kk, rs in by_img.items():\n        gts = gt_boxes(rs)\n        cb = [d[:4] for d in cached.get(kk, []) if d[4] >= DK.DET_SCORE]\n        nb = [d[:4] for d in new.get(kk, []) if d[4] >= DK.DET_SCORE]\n        mc, mn = DK.match(gts, cb), DK.match(gts, nb)\n        if set(mc) == set(mn):\n            same += len(rs)\n        else:\n            diff.append({\"image\": kk, \"only_cached\": sorted(set(mc) - set(mn)), \"only_new\": sorted(set(mn) - set(mc))})\n        for i in set(mc) & set(mn):\n            ious.append(DK.box_iou(cb[mc[i][0]], nb[mn[i][0]]))\n    med = float(np.median(ious)) if ious else 0.0\n    return {\"images\": len(by_img), \"findings\": len(rows), \"findings_same_match\": same, \"images_with_different_match\": diff,\n            \"median_box_iou\": med, \"pass\": (not diff) and med >= DET_IOU_MEDIAN_MIN}\n\n\n# ------------------------------------------------------------------------------------------------------------- evaluation\ndef macro_f1(y, pred, n=len(CLASSES)) -> float:\n    \"\"\"Same definition as DK.report (vectorised for the bootstrap).\"\"\"\n    cm = np.bincount(np.asarray(y) * n + np.asarray(pred), minlength=n * n).reshape(n, n)\n    tp, pp, sup = np.diag(cm), cm.sum(0), cm.sum(1)\n    prec = np.divide(tp, pp, out=np.zeros(n), where=pp > 0)\n    rec = np.divide(tp, sup, out=np.zeros(n), where=sup > 0)\n    return float(np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0).mean())\n\n\ndef check_oof(oof: dict, n_findings: int, seeds=SEEDS) -> None:\n    \"\"\"Gate 5: oof[seed]['gt'] is (N, C) with every row filled exactly once (no NaN), for every seed.\"\"\"\n    for s in seeds:\n        if s not in oof:\n            raise CVError(f\"seed {s}: no out-of-fold predictions\")\n        p = oof[s][\"gt\"]\n        if p.shape != (n_findings, len(CLASSES)) or np.isnan(p).any():\n            raise CVError(f\"seed {s}: out-of-fold GT predictions incomplete ({int(np.isnan(p).any(1).sum())} findings missing)\")\n        if (oof[s][\"filled\"] != 1).any():\n            raise CVError(f\"seed {s}: {int((oof[s]['filled'] != 1).sum())} findings predicted 0 or >1 times\")\n\n\ndef pooled_reports(y, oof: dict, matched: np.ndarray, seeds=SEEDS, subset=None) -> dict:\n    \"\"\"Per seed: DK.report on pooled out-of-fold predictions; detector setting on matched findings only; end-to-end recall.\"\"\"\n    idx = np.arange(len(y)) if subset is None else np.asarray(subset)\n    out = {}\n    for s in seeds:\n        pg, pdet = oof[s][\"gt\"][idx], oof[s][\"det\"][idx]\n        m = matched[idx]\n        r = {\"gt\": DK.report(y[idx], pg.argmax(1), pg)}\n        if m.any():\n            r[\"det_matched\"] = DK.report(y[idx][m], pdet[m].argmax(1), pdet[m])\n        e2e = np.where(m, np.nan_to_num(pdet, nan=0).argmax(1), -1)\n        r[\"end_to_end\"] = {\"accuracy\": float((e2e == y[idx]).mean()), \"unmatched_findings\": int((~m).sum()),\n                           \"recall\": {c: float(((e2e == k) & (y[idx] == k)).sum() / max(1, (y[idx] == k).sum())) for k, c in enumerate(CLASSES)}}\n        out[s] = r\n    return out\n\n\ndef cluster_bootstrap(y, preds_by_seed: list, groups, valid=None, n_boot=BOOT_N, seed=BOOT_SEED) -> dict:\n    \"\"\"95 % percentile interval of the seed-mean macro-F1, resampling RADIOGRAPHS (all their findings) with replacement.\n    valid: boolean mask of findings that count (e.g. detector-matched); invalid findings are dropped after resampling.\"\"\"\n    y = np.asarray(y)\n    groups = np.asarray(groups)\n    valid = np.ones(len(y), bool) if valid is None else np.asarray(valid)\n    uniq = np.unique(groups)\n    members = {g: np.flatnonzero(groups == g) for g in uniq}\n    rng = np.random.default_rng(seed)\n    stats = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.choice(uniq, size=len(uniq), replace=True)\n        i = np.concatenate([members[g] for g in pick])\n        i = i[valid[i]]\n        stats[b] = np.mean([macro_f1(y[i], np.asarray(p)[i]) for p in preds_by_seed])\n    point = float(np.mean([macro_f1(y[valid], np.asarray(p)[valid]) for p in preds_by_seed]))\n    return {\"unit\": \"radiograph\", \"n_radiographs\": int(len(uniq)), \"n_boot\": n_boot, \"seed\": seed, \"point\": point,\n            \"ci95\": [float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))], \"boot_sd\": float(stats.std(ddof=1))}\n\n\ndef mean_sd(values) -> dict:\n    v = [float(x) for x in values]\n    return {\"values\": v, \"mean\": float(np.mean(v)), \"sd\": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0}\n\n\ndef tooth_type(fdi) -> str:\n    try:\n        d = int(str(fdi)[-1])\n    except (TypeError, ValueError):\n        return \"unknown\"\n    return {1: \"incisor\", 2: \"incisor\", 3: \"canine\", 4: \"premolar\", 5: \"premolar\"}.get(d, \"molar\" if d in (6, 7, 8) else \"unknown\")\n\n\ndef caries_confusion(rows, oof: dict, matched: np.ndarray, seeds=SEEDS) -> dict:\n    \"\"\"Exploratory (protocol section 6). For every Caries / Deep Caries finding: how many seeds predicted the other caries class,\n    per setting. Returns per-finding records plus summaries by tooth type and box-area quartile. Labels are not judged.\"\"\"\n    C, DC = CLASSES.index(\"Caries\"), CLASSES.index(\"Deep Caries\")\n    y = np.array([int(r[\"class_index\"]) for r in rows])\n    sel = np.flatnonzero((y == C) | (y == DC))\n    other = np.where(y == C, DC, C)\n    n_gt = np.zeros(len(y), int)\n    n_det = np.zeros(len(y), int)\n    for s in seeds:\n        n_gt += oof[s][\"gt\"].argmax(1) == other\n        n_det += np.where(matched, np.nan_to_num(oof[s][\"det\"], nan=0).argmax(1) == other, False)\n    area = np.array([np.prod(json.loads(r[\"bbox_xywh\"])[2:]) for r in rows], float)\n    q = np.quantile(area[sel], [0.25, 0.5, 0.75])\n    quart = np.searchsorted(q, area, side=\"right\") + 1\n    recs = []\n    for i in sel:\n        r = rows[i]\n        recs.append({\"sample_id\": r[\"sample_id\"], \"image_path\": r[\"image_path\"], \"radiograph_group\": r[\"radiograph_group\"],\n                     \"fdi_tooth\": r[\"fdi_tooth\"], \"tooth_type\": tooth_type(r[\"fdi_tooth\"]), \"label\": r[\"target_class\"],\n                     \"original_label\": r[\"original_label\"], \"box_area_px\": int(area[i]), \"box_area_quartile\": int(quart[i]),\n                     \"detector_matched\": bool(matched[i]), \"seeds_other_caries_gt\": int(n_gt[i]),\n                     \"seeds_other_caries_det\": int(n_det[i]) if matched[i] else \"\"})\n    def summary(key):\n        out = {}\n        for lab in (\"Caries\", \"Deep Caries\"):\n            for v in sorted({rr[key] for rr in recs}):\n                grp = [rr for rr in recs if rr[\"label\"] == lab and rr[key] == v]\n                if grp:\n                    out[f\"{lab} | {key}={v}\"] = {\"n\": len(grp), \"mean_share_of_seeds_other_caries_gt\":\n                                                 float(np.mean([rr[\"seeds_other_caries_gt\"] for rr in grp]) / len(seeds)),\n                                                 \"confused_by_all_seeds_gt\": sum(rr[\"seeds_other_caries_gt\"] == len(seeds) for rr in grp)}\n        return out\n    return {\"records\": recs, \"box_area_quartile_edges_px\": [float(v) for v in q],\n            \"by_tooth_type\": summary(\"tooth_type\"), \"by_box_area_quartile\": summary(\"box_area_quartile\"),\n            \"confused_by_all_seeds_gt\": {lab: sum(rr[\"label\"] == lab and rr[\"seeds_other_caries_gt\"] == len(seeds) for rr in recs)\n                                         for lab in (\"Caries\", \"Deep Caries\")},\n            \"n\": {lab: sum(rr[\"label\"] == lab for rr in recs) for lab in (\"Caries\", \"Deep Caries\")}}\n", "scripts/dataset/make_cv_folds.py": "#!/usr/bin/env python\r\n\"\"\"P4 (approved with review gate, 2026-10-16): DRAFT grouped 5-fold assignment for future experiments. Nothing runs from this file\r\nuntil the owner has reviewed and approved the exact assignment.\r\n\r\n    .venv\\\\Scripts\\\\python.exe scripts\\\\dataset\\\\make_cv_folds.py\r\n\r\n* Units are radiograph groups (all crops of one radiograph are in one fold): data/splits/combined_train.csv + combined_validation.csv.\r\n  The frozen test split and the calibration split are excluded.\r\n* Stratified by (source family, has Periapical Lesion, has Deep Caries); within a stratum, groups are taken in a seeded random order\r\n  and given to the fold with the fewest groups of that stratum (ties: fewest findings, then lowest fold number).\r\n* Writes data/splits/cv5_folds_DRAFT.csv and reports/cv5_fold_draft_summary.md (per-fold radiographs and findings per class / source).\r\n  Adding verified labels changes the input, so the draft must be regenerated and reviewed again.\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport argparse\r\nimport hashlib\r\nimport random\r\nimport sys\r\nfrom collections import Counter, defaultdict\r\nfrom pathlib import Path\r\n\r\nsys.path.insert(0, str(Path(__file__).resolve().parent))\r\nfrom dsutil import CLASSES, read_csv, write_csv  # noqa: E402\r\n\r\nROOT = Path(__file__).resolve().parents[2]\r\nDATA = ROOT / \"data\"\r\nK, SEED = 5, 42\r\n\r\n\r\ndef assign_folds(rows, k=K, seed=SEED, with_impacted=False):\r\n    \"\"\"rows: manifest rows -> {radiograph_group: fold}. with_impacted=True adds \"has Impacted\" to the stratum (option B of\r\n    reports/cv5_fold_alternative_comparison.md, approved by the owner on 2026-10-24); False reproduces the reviewed draft.\"\"\"\r\n    g = defaultdict(list)\r\n    for r in rows:\r\n        g[r[\"radiograph_group\"]].append(r)\r\n    def stratum(rs):\r\n        src = rs[0][\"source_dataset\"].split(\":\")[0]\r\n        cls = {r[\"target_class\"] for r in rs}\r\n        key = (src, \"Periapical Lesion\" in cls, \"Deep Caries\" in cls)\r\n        return key + (\"Impacted\" in cls,) if with_impacted else key\r\n    strata = defaultdict(list)\r\n    for gid, rs in g.items():\r\n        strata[stratum(rs)].append(gid)\r\n    fold_of, per_stratum, findings = {}, defaultdict(Counter), Counter()\r\n    rng = random.Random(seed)\r\n    for key in sorted(strata, key=str):\r\n        gids = sorted(strata[key])\r\n        rng.shuffle(gids)\r\n        for gid in gids:\r\n            f = min(range(k), key=lambda i: (per_stratum[key][i], findings[i], i))\r\n            fold_of[gid] = f\r\n            per_stratum[key][f] += 1\r\n            findings[f] += len(g[gid])\r\n    return fold_of\r\n\r\n\r\ndef main():\r\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\r\n    ap.add_argument(\"--approved-v1\", action=\"store_true\",\r\n                    help=\"write the owner-approved assignment (option B, Impacted-stratified) to data/splits/cv5_folds_v1.csv\")\r\n    a = ap.parse_args()\r\n    rows = read_csv(DATA / \"splits/combined_train.csv\") + read_csv(DATA / \"splits/combined_validation.csv\")\r\n    test_groups = {r[\"radiograph_group\"] for r in read_csv(DATA / \"splits/combined_test.csv\") + read_csv(DATA / \"splits/combined_calibration.csv\")}\r\n    assert not ({r[\"radiograph_group\"] for r in rows} & test_groups), \"test / calibration radiograph in the CV pool\"\r\n    fold_of = assign_folds(rows, with_impacted=a.approved_v1)\r\n    out = []\r\n    for gid in sorted(fold_of):\r\n        rs = [r for r in rows if r[\"radiograph_group\"] == gid]\r\n        c = Counter(r[\"target_class\"] for r in rs)\r\n        out.append({\"radiograph_group\": gid, \"fold\": fold_of[gid], \"source\": rs[0][\"source_dataset\"],\r\n                    \"original_split\": rs[0][\"split\"], \"n_findings\": len(rs), **{c_.replace(\" \", \"_\"): c[c_] for c_ in CLASSES}})\r\n    p = DATA / (\"splits/cv5_folds_v1.csv\" if a.approved_v1 else \"splits/cv5_folds_DRAFT.csv\")\r\n    write_csv(p, out)\r\n    sha = hashlib.sha256(p.read_bytes()).hexdigest()\r\n    title = (\"# APPROVED grouped 5-fold assignment v1 (P4, option B: Impacted-stratified; owner approval 2026-10-24)\" if a.approved_v1\r\n             else \"# DRAFT grouped 5-fold assignment (P4): FOR REVIEW, NOT APPROVED, NOT USED\")\r\n    lines = [title, \"\",\r\n             f\"File `data/splits/{p.name}`, SHA-256 `{sha}`, seed {SEED}. Units: radiograph groups of combined_train + \"\r\n             \"combined_validation; test and calibration excluded (checked: no overlap).\", \"\",\r\n             \"| Fold | Radiographs | Findings | Caries | Deep Caries | Impacted | Periapical Lesion | PL radiographs | DC radiographs | from old val split |\",\r\n             \"|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|\"]\r\n    for f in range(K):\r\n        fr = [o for o in out if o[\"fold\"] == f]\r\n        lines.append(f\"| {f} | {len(fr)} | {sum(o['n_findings'] for o in fr)} | \" + \" | \".join(str(sum(o[c.replace(' ', '_')] for o in fr)) for c in CLASSES)\r\n                     + f\" | {sum(1 for o in fr if o['Periapical_Lesion'])} | {sum(1 for o in fr if o['Deep_Caries'])}\"\r\n                     + f\" | {sum(1 for o in fr if o['original_split'] == 'validation')} |\")\r\n    src = Counter((o[\"fold\"], o[\"source\"]) for o in out)\r\n    lines += [\"\", \"Radiographs per fold and source: \" + \"; \".join(f\"fold {f}: \" + \", \".join(f\"{s} {n}\" for (ff, s), n in sorted(src.items()) if ff == f)\r\n                                                        for f in range(K)),\r\n              \"\", \"Review points: (1) every fold has Periapical Lesion radiographs; (2) source mix per fold; (3) the old validation \"\r\n              \"radiographs are spread over folds (they are trained on in 4 of 5 folds); (4) after clinician-verified labels are added, this \"\r\n              \"draft is regenerated and needs a new review.\"]\r\n    (ROOT / (\"reports/cv5_folds_v1_summary.md\" if a.approved_v1 else \"reports/cv5_fold_draft_summary.md\")).write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\r\n    print(\"\\n\".join(lines))\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    main()\r\n", "data/splits/cv5_folds_v1.csv": "radiograph_group,fold,source,original_split,n_findings,Caries,Deep_Caries,Impacted,Periapical_Lesion\r\ng00092,1,dentex_train_disease,train,8,5,3,0,0\r\ng00252,2,dentex_train_disease,train,5,5,0,0,0\r\ng00253,1,dentex_train_disease,train,9,6,0,1,2\r\ng00256,0,dentex_train_disease,validation,4,1,0,3,0\r\ng00257,2,dentex_train_disease,validation,12,9,2,1,0\r\ng00258,4,dentex_train_disease,train,8,5,0,3,0\r\ng00259,3,dentex_train_disease,train,1,1,0,0,0\r\ng00260,0,dentex_train_disease,train,4,4,0,0,0\r\ng00262,1,dentex_train_disease,train,9,7,0,2,0\r\ng00263,4,dentex_train_disease,train,8,6,1,0,1\r\ng00264,3,dentex_train_disease,train,6,1,1,4,0\r\ng00265,4,dentex_train_disease,train,1,1,0,0,0\r\ng00266,1,dentex_train_disease,train,4,4,0,0,0\r\ng00267,3,dentex_train_disease,train,2,1,1,0,0\r\ng00268,3,dentex_train_disease,train,1,1,0,0,0\r\ng00269,1,dentex_train_disease,train,1,1,0,0,0\r\ng00270,4,dentex_train_disease,train,1,1,0,0,0\r\ng00271,0,dentex_train_disease,train,5,3,0,1,1\r\ng00273,2,dentex_train_disease,train,2,2,0,0,0\r\ng00274,1,dentex_train_disease,validation,8,6,2,0,0\r\ng00276,4,dentex_train_disease,train,7,6,1,0,0\r\ng00280,1,dentex_train_disease,validation,3,3,0,0,0\r\ng00281,1,dentex_train_disease,train,5,5,0,0,0\r\ng00282,1,dentex_train_disease,train,13,6,3,4,0\r\ng00283,3,dentex_train_disease,train,6,2,1,3,0\r\ng00284,0,dentex_train_disease,train,2,0,2,0,0\r\ng00286,0,dentex_train_disease,train,6,2,0,4,0\r\ng00287,4,dentex_train_disease,train,3,2,1,0,0\r\ng00288,3,dentex_train_disease,validation,2,1,0,1,0\r\ng00289,2,dentex_train_disease,train,6,6,0,0,0\r\ng00290,0,dentex_train_disease,train,11,3,4,4,0\r\ng00291,0,dentex_train_disease,train,8,6,2,0,0\r\ng00292,1,dentex_train_disease,train,6,5,1,0,0\r\ng00293,0,dentex_train_disease,train,5,2,3,0,0\r\ng00294,3,dentex_train_disease,train,7,1,5,0,1\r\ng00296,4,dentex_train_disease,train,8,4,2,1,1\r\ng00297,1,dentex_train_disease,validation,1,1,0,0,0\r\ng00298,0,dentex_train_disease,train,5,2,3,0,0\r\ng00299,2,dentex_train_disease,train,6,6,0,0,0\r\ng00300,1,dentex_train_disease,train,15,11,0,0,4\r\ng00302,2,dentex_train_disease,train,5,4,1,0,0\r\ng00303,3,dentex_train_disease,train,10,3,3,4,0\r\ng00304,2,dentex_train_disease,train,1,1,0,0,0\r\ng00305,4,dentex_train_disease,train,1,1,0,0,0\r\ng00307,2,dentex_train_disease,train,8,3,3,2,0\r\ng00308,3,dentex_train_disease,train,6,5,0,1,0\r\ng00309,2,dentex_train_disease,validation,5,4,0,1,0\r\ng00310,2,dentex_train_disease,train,2,2,0,0,0\r\ng00311,2,dentex_train_disease,train,6,4,2,0,0\r\ng00312,0,dentex_train_disease,train,4,3,1,0,0\r\ng00313,1,dentex_train_disease,train,2,1,0,1,0\r\ng00314,1,dentex_train_disease,train,5,4,1,0,0\r\ng00315,2,dentex_train_disease,train,15,9,5,0,1\r\ng00317,3,dentex_train_disease,validation,1,1,0,0,0\r\ng00318,0,dentex_train_disease,train,7,3,0,4,0\r\ng00319,2,dentex_train_disease,train,2,0,1,0,1\r\ng00320,2,dentex_train_disease,train,1,1,0,0,0\r\ng00322,3,dentex_train_disease,train,2,2,0,0,0\r\ng00323,2,dentex_train_disease,validation,7,6,1,0,0\r\ng00324,0,dentex_train_disease,train,1,1,0,0,0\r\ng00325,0,dentex_train_disease,train,11,7,0,0,4\r\ng00326,4,dentex_train_disease,train,5,5,0,0,0\r\ng00327,3,dentex_train_disease,validation,2,1,0,1,0\r\ng00328,4,dentex_train_disease,train,6,4,0,2,0\r\ng00329,0,dentex_train_disease,train,11,9,2,0,0\r\ng00330,1,dentex_train_disease,train,8,6,1,1,0\r\ng00331,4,dentex_train_disease,train,1,0,0,1,0\r\ng00333,2,dentex_train_disease,train,7,5,2,0,0\r\ng00334,3,dentex_train_disease,train,4,2,1,0,1\r\ng00335,4,dentex_train_disease,train,4,4,0,0,0\r\ng00336,4,dentex_train_disease,train,8,4,0,4,0\r\ng00337,2,dentex_train_disease,validation,10,6,0,4,0\r\ng00338,3,dentex_train_disease,train,4,3,1,0,0\r\ng00341,2,dentex_train_disease,train,2,1,0,1,0\r\ng00342,3,dentex_train_disease,train,3,1,2,0,0\r\ng00343,2,dentex_train_disease,train,11,3,8,0,0\r\ng00344,2,dentex_train_disease,train,1,1,0,0,0\r\ng00345,2,dentex_train_disease,train,11,2,4,4,1\r\ng00347,3,dentex_train_disease,train,4,4,0,0,0\r\ng00348,0,dentex_train_disease,train,9,6,3,0,0\r\ng00349,2,dentex_train_disease,train,15,7,8,0,0\r\ng00350,3,dentex_train_disease,train,9,3,1,3,2\r\ng00351,4,dentex_train_disease,train,11,7,0,4,0\r\ng00352,4,dentex_train_disease,validation,4,1,3,0,0\r\ng00353,0,dentex_train_disease,train,6,4,1,1,0\r\ng00354,3,dentex_train_disease,train,8,3,1,4,0\r\ng00355,1,dentex_train_disease,train,5,4,1,0,0\r\ng00356,0,dentex_train_disease,train,6,5,0,0,1\r\ng00357,4,dentex_train_disease,train,6,2,0,4,0\r\ng00358,1,dentex_train_disease,train,1,0,0,1,0\r\ng00359,4,dentex_train_disease,train,4,2,0,2,0\r\ng00360,0,dentex_train_disease,train,2,1,1,0,0\r\ng00361,0,dentex_train_disease,train,8,4,4,0,0\r\ng00362,3,dentex_train_disease,train,10,5,4,1,0\r\ng00364,1,dentex_train_disease,train,1,1,0,0,0\r\ng00365,2,dentex_train_disease,train,2,1,0,1,0\r\ng00366,2,dentex_train_disease,train,2,1,1,0,0\r\ng00367,0,dentex_train_disease,validation,4,4,0,0,0\r\ng00368,2,dentex_train_disease,train,4,4,0,0,0\r\ng00371,3,dentex_train_disease,train,2,1,1,0,0\r\ng00373,0,dentex_train_disease,train,6,1,2,3,0\r\ng00374,0,dentex_train_disease,train,3,1,1,0,1\r\ng00375,1,dentex_train_disease,train,5,3,2,0,0\r\ng00377,2,dentex_train_disease,train,5,1,0,4,0\r\ng00379,3,dentex_train_disease,train,4,3,1,0,0\r\ng00380,2,dentex_train_disease,train,6,5,1,0,0\r\ng00381,4,dentex_train_disease,validation,2,1,0,1,0\r\ng00382,2,dentex_train_disease,train,7,3,0,4,0\r\ng00383,1,dentex_train_disease,validation,4,3,1,0,0\r\ng00386,0,dentex_train_disease,train,3,0,1,2,0\r\ng00389,4,dentex_train_disease,train,1,0,0,0,1\r\ng00390,3,dentex_train_disease,train,1,1,0,0,0\r\ng00392,1,dentex_train_disease,train,1,1,0,0,0\r\ng00393,0,dentex_train_disease,train,6,3,0,3,0\r\ng00394,0,dentex_train_disease,train,2,2,0,0,0\r\ng00397,2,dentex_train_disease,train,9,9,0,0,0\r\ng00398,4,dentex_train_disease,validation,6,2,0,4,0\r\ng00399,1,dentex_train_disease,train,5,5,0,0,0\r\ng00400,2,dentex_train_disease,train,1,0,0,0,1\r\ng00401,2,dentex_train_disease,train,6,5,1,0,0\r\ng00402,2,dentex_train_disease,train,1,1,0,0,0\r\ng00403,3,dentex_train_disease,train,5,0,0,4,1\r\ng00404,1,dentex_train_disease,train,7,5,1,0,1\r\ng00406,2,dentex_train_disease,train,3,2,0,0,1\r\ng00407,0,dentex_train_disease,validation,9,7,2,0,0\r\ng00408,1,dentex_train_disease,train,7,3,0,4,0\r\ng00409,4,dentex_train_disease,validation,4,1,3,0,0\r\ng00410,4,dentex_train_disease,train,9,5,2,2,0\r\ng00411,0,dentex_train_disease,validation,2,1,0,1,0\r\ng00412,1,dentex_train_disease,train,6,2,4,0,0\r\ng00413,0,dentex_train_disease,train,7,7,0,0,0\r\ng00415,0,dentex_train_disease,train,3,2,1,0,0\r\ng00416,4,dentex_train_disease,train,5,4,1,0,0\r\ng00418,0,dentex_train_disease,train,9,5,0,4,0\r\ng00419,3,dentex_train_disease,train,5,4,0,1,0\r\ng00421,2,dentex_train_disease,train,6,6,0,0,0\r\ng00423,2,dentex_train_disease,train,11,10,0,1,0\r\ng00424,1,dentex_train_disease,train,8,6,1,0,1\r\ng00425,0,dentex_train_disease,train,2,1,1,0,0\r\ng00426,4,dentex_train_disease,train,9,4,1,4,0\r\ng00429,3,dentex_train_disease,train,5,2,1,2,0\r\ng00430,3,dentex_train_disease,train,9,0,4,4,1\r\ng00431,3,dentex_train_disease,train,7,3,0,4,0\r\ng00432,2,dentex_train_disease,train,6,5,0,1,0\r\ng00433,1,dentex_train_disease,train,10,4,1,4,1\r\ng00437,0,dentex_train_disease,train,23,14,3,4,2\r\ng00438,3,dentex_train_disease,train,1,1,0,0,0\r\ng00439,3,dentex_train_disease,train,4,0,2,0,2\r\ng00440,2,dentex_train_disease,train,2,2,0,0,0\r\ng00441,2,dentex_train_disease,train,5,4,0,1,0\r\ng00442,4,dentex_train_disease,train,3,3,0,0,0\r\ng00443,3,dentex_train_disease,validation,9,5,0,4,0\r\ng00444,1,dentex_train_disease,train,3,2,1,0,0\r\ng00445,0,dentex_train_disease,train,15,10,3,0,2\r\ng00446,2,dentex_train_disease,validation,8,4,0,2,2\r\ng00447,3,dentex_train_disease,train,10,6,0,4,0\r\ng00449,1,dentex_train_disease,train,10,5,1,4,0\r\ng00450,2,dentex_train_disease,validation,2,1,1,0,0\r\ng00451,2,dentex_train_disease,train,3,3,0,0,0\r\ng00452,1,dentex_train_disease,train,1,0,1,0,0\r\ng00453,3,dentex_train_disease,train,3,3,0,0,0\r\ng00454,2,dentex_train_disease,train,13,10,3,0,0\r\ng00456,4,dentex_train_disease,train,2,1,1,0,0\r\ng00457,3,dentex_train_disease,train,8,5,3,0,0\r\ng00458,3,dentex_train_disease,validation,6,2,0,4,0\r\ng00459,1,dentex_train_disease,train,5,2,2,0,1\r\ng00460,0,dentex_train_disease,train,10,1,7,0,2\r\ng00461,0,dentex_train_disease,train,5,3,0,2,0\r\ng00462,0,dentex_train_disease,train,1,1,0,0,0\r\ng00464,4,dentex_train_disease,train,8,8,0,0,0\r\ng00465,0,dentex_train_disease,train,1,1,0,0,0\r\ng00466,2,dentex_train_disease,train,21,10,9,2,0\r\ng00467,4,dentex_train_disease,train,5,1,2,2,0\r\ng00468,2,dentex_train_disease,train,7,3,0,4,0\r\ng00469,1,dentex_train_disease,train,10,4,4,0,2\r\ng00472,1,dentex_train_disease,validation,3,3,0,0,0\r\ng00473,1,dentex_train_disease,train,3,2,1,0,0\r\ng00474,0,dentex_train_disease,train,1,0,0,0,1\r\ng00475,4,dentex_train_disease,train,4,4,0,0,0\r\ng00476,4,dentex_train_disease,validation,4,4,0,0,0\r\ng00477,4,dentex_train_disease,validation,4,2,0,2,0\r\ng00478,2,dentex_train_disease,train,4,4,0,0,0\r\ng00479,3,dentex_train_disease,train,6,6,0,0,0\r\ng00480,0,dentex_train_disease,train,3,3,0,0,0\r\ng00481,1,dentex_train_disease,train,1,1,0,0,0\r\ng00483,4,dentex_train_disease,train,4,2,0,2,0\r\ng00484,0,dentex_train_disease,train,2,2,0,0,0\r\ng00485,2,dentex_train_disease,train,4,3,1,0,0\r\ng00486,4,dentex_train_disease,train,2,1,1,0,0\r\ng00488,1,dentex_train_disease,train,3,3,0,0,0\r\ng00489,4,dentex_train_disease,train,1,0,0,1,0\r\ng00490,0,dentex_train_disease,train,6,6,0,0,0\r\ng00491,3,dentex_train_disease,train,9,4,1,4,0\r\ng00492,3,dentex_train_disease,train,6,5,1,0,0\r\ng00493,3,dentex_train_disease,train,1,0,0,0,1\r\ng00495,3,dentex_train_disease,train,3,3,0,0,0\r\ng00497,3,dentex_train_disease,train,2,2,0,0,0\r\ng00498,2,dentex_train_disease,train,3,2,1,0,0\r\ng00499,2,dentex_train_disease,train,2,2,0,0,0\r\ng00500,1,dentex_train_disease,validation,1,0,1,0,0\r\ng00501,0,dentex_train_disease,train,1,0,1,0,0\r\ng00504,1,dentex_train_disease,train,12,10,2,0,0\r\ng00505,2,dentex_train_disease,validation,11,5,2,4,0\r\ng00506,1,dentex_train_disease,validation,4,2,2,0,0\r\ng00507,1,dentex_train_disease,train,4,4,0,0,0\r\ng00508,4,dentex_train_disease,train,5,3,0,2,0\r\ng00510,3,dentex_train_disease,train,12,10,0,1,1\r\ng00511,1,dentex_train_disease,train,3,3,0,0,0\r\ng00512,0,dentex_train_disease,train,5,5,0,0,0\r\ng00515,3,dentex_train_disease,train,3,2,1,0,0\r\ng00516,4,dentex_train_disease,train,6,2,0,4,0\r\ng00517,4,dentex_train_disease,train,1,1,0,0,0\r\ng00518,0,dentex_train_disease,train,9,6,0,3,0\r\ng00519,2,dentex_train_disease,train,9,7,1,0,1\r\ng00520,0,dentex_train_disease,train,2,1,0,1,0\r\ng00521,4,dentex_train_disease,train,3,2,1,0,0\r\ng00523,1,dentex_train_disease,validation,3,2,1,0,0\r\ng00526,3,dentex_train_disease,train,3,3,0,0,0\r\ng00527,4,dentex_train_disease,train,8,2,2,4,0\r\ng00528,3,dentex_train_disease,train,12,11,1,0,0\r\ng00529,0,dentex_train_disease,train,8,4,2,2,0\r\ng00530,1,dentex_train_disease,train,7,3,0,4,0\r\ng00531,2,dentex_train_disease,train,6,4,2,0,0\r\ng00532,1,dentex_train_disease,train,4,3,1,0,0\r\ng00533,0,dentex_train_disease,train,10,5,1,4,0\r\ng00534,1,dentex_train_disease,train,1,0,0,0,1\r\ng00536,4,dentex_train_disease,train,7,5,1,0,1\r\ng00538,3,dentex_train_disease,train,2,1,1,0,0\r\ng00540,3,dentex_train_disease,validation,1,1,0,0,0\r\ng00541,1,dentex_train_disease,train,9,6,3,0,0\r\ng00542,1,dentex_train_disease,train,8,6,2,0,0\r\ng00544,0,dentex_train_disease,train,7,3,0,4,0\r\ng00547,2,dentex_train_disease,train,3,2,1,0,0\r\ng00549,4,dentex_train_disease,train,3,3,0,0,0\r\ng00550,4,dentex_train_disease,train,5,3,1,1,0\r\ng00551,4,dentex_train_disease,train,5,5,0,0,0\r\ng00552,2,dentex_train_disease,train,1,0,1,0,0\r\ng00553,1,dentex_train_disease,train,2,1,1,0,0\r\ng00555,0,dentex_train_disease,train,9,8,1,0,0\r\ng00557,4,dentex_train_disease,train,9,8,1,0,0\r\ng00558,2,dentex_train_disease,train,5,2,3,0,0\r\ng00560,3,dentex_train_disease,train,5,5,0,0,0\r\ng00561,0,dentex_train_disease,validation,1,1,0,0,0\r\ng00562,1,dentex_train_disease,train,4,2,1,0,1\r\ng00563,4,dentex_train_disease,train,3,2,1,0,0\r\ng00564,3,dentex_train_disease,train,17,13,1,1,2\r\ng00565,3,dentex_train_disease,train,1,0,0,0,1\r\ng00566,4,dentex_train_disease,train,6,3,3,0,0\r\ng00567,0,dentex_train_disease,train,2,1,1,0,0\r\ng00568,2,dentex_train_disease,train,1,0,0,0,1\r\ng00569,2,dentex_train_disease,train,11,7,0,4,0\r\ng00571,2,dentex_train_disease,train,1,0,0,1,0\r\ng00572,3,dentex_train_disease,train,4,1,0,3,0\r\ng00573,3,dentex_train_disease,train,2,1,1,0,0\r\ng00574,4,dentex_train_disease,validation,3,3,0,0,0\r\ng00575,4,dentex_train_disease,train,3,2,1,0,0\r\ng00576,4,dentex_train_disease,train,2,1,0,1,0\r\ng00577,0,dentex_train_disease,train,7,7,0,0,0\r\ng00579,0,dentex_train_disease,train,6,6,0,0,0\r\ng00580,3,dentex_train_disease,train,5,4,0,0,1\r\ng00581,1,dentex_train_disease,train,7,0,5,1,1\r\ng00582,2,dentex_train_disease,train,1,0,1,0,0\r\ng00584,0,dentex_train_disease,train,5,2,3,0,0\r\ng00585,1,dentex_train_disease,train,9,5,3,0,1\r\ng00586,0,dentex_train_disease,train,2,2,0,0,0\r\ng00587,3,dentex_train_disease,train,8,6,1,0,1\r\ng00589,3,dentex_train_disease,train,3,2,1,0,0\r\ng00590,0,dentex_train_disease,train,4,1,0,3,0\r\ng00591,1,dentex_train_disease,train,5,4,0,1,0\r\ng00592,4,dentex_train_disease,train,9,7,1,1,0\r\ng00594,4,dentex_train_disease,train,8,8,0,0,0\r\ng00596,4,dentex_train_disease,train,3,3,0,0,0\r\ng00597,0,dentex_train_disease,train,6,3,2,1,0\r\ng00598,4,dentex_train_disease,train,7,1,2,4,0\r\ng00599,3,dentex_train_disease,train,2,2,0,0,0\r\ng00600,0,dentex_train_disease,train,3,2,1,0,0\r\ng00601,4,dentex_train_disease,train,4,3,1,0,0\r\ng00602,0,dentex_train_disease,train,3,3,0,0,0\r\ng00603,0,dentex_train_disease,train,7,5,0,1,1\r\ng00604,0,dentex_train_disease,train,9,7,0,2,0\r\ng00605,4,dentex_train_disease,train,4,3,0,1,0\r\ng00606,1,dentex_train_disease,validation,6,4,0,2,0\r\ng00608,0,dentex_train_disease,train,4,2,1,0,1\r\ng00609,0,dentex_train_disease,train,2,2,0,0,0\r\ng00610,4,dentex_train_disease,validation,1,1,0,0,0\r\ng00611,3,dentex_train_disease,train,3,1,0,2,0\r\ng00612,0,dentex_train_disease,train,5,1,0,4,0\r\ng00613,3,dentex_train_disease,train,4,4,0,0,0\r\ng00614,2,dentex_train_disease,train,2,2,0,0,0\r\ng00615,2,dentex_train_disease,train,3,1,2,0,0\r\ng00616,4,dentex_train_disease,train,4,3,1,0,0\r\ng00617,3,dentex_train_disease,train,3,1,0,2,0\r\ng00618,1,dentex_train_disease,validation,3,3,0,0,0\r\ng00619,1,dentex_train_disease,train,6,3,1,2,0\r\ng00621,3,dentex_train_disease,validation,7,5,0,2,0\r\ng00623,3,dentex_train_disease,train,3,2,1,0,0\r\ng00624,3,dentex_train_disease,validation,3,3,0,0,0\r\ng00625,2,dentex_train_disease,train,1,0,1,0,0\r\ng00626,3,dentex_train_disease,train,5,5,0,0,0\r\ng00627,4,dentex_train_disease,train,9,5,3,1,0\r\ng00628,2,dentex_train_disease,train,6,1,3,1,1\r\ng00629,4,dentex_train_disease,train,1,1,0,0,0\r\ng00631,4,dentex_train_disease,train,1,0,0,0,1\r\ng00632,2,dentex_train_disease,train,6,4,2,0,0\r\ng00633,0,dentex_train_disease,train,2,2,0,0,0\r\ng00634,4,dentex_train_disease,train,5,4,1,0,0\r\ng00636,0,dentex_train_disease,train,5,2,0,3,0\r\ng00639,0,dentex_train_disease,validation,16,10,4,0,2\r\ng00641,3,dentex_train_disease,validation,4,3,1,0,0\r\ng00642,0,dentex_train_disease,train,4,3,1,0,0\r\ng00643,1,dentex_train_disease,train,4,3,0,1,0\r\ng00644,3,dentex_train_disease,train,8,7,0,0,1\r\ng00646,4,dentex_train_disease,train,1,0,0,0,1\r\ng00647,3,dentex_train_disease,train,3,2,0,1,0\r\ng00648,2,dentex_train_disease,train,7,4,1,2,0\r\ng00649,3,dentex_train_disease,train,7,1,1,4,1\r\ng00651,4,dentex_train_disease,train,7,5,1,0,1\r\ng00652,2,dentex_train_disease,validation,2,0,1,0,1\r\ng00653,0,dentex_train_disease,train,3,2,0,0,1\r\ng00654,1,dentex_train_disease,train,14,11,3,0,0\r\ng00655,4,dentex_train_disease,train,7,7,0,0,0\r\ng00656,2,dentex_train_disease,train,10,6,1,1,2\r\ng00658,3,dentex_train_disease,train,12,12,0,0,0\r\ng00659,1,dentex_train_disease,train,14,10,3,1,0\r\ng00660,1,dentex_train_disease,train,4,1,0,3,0\r\ng00661,0,dentex_train_disease,train,4,0,0,4,0\r\ng00663,0,dentex_train_disease,train,5,4,1,0,0\r\ng00664,1,dentex_train_disease,train,2,2,0,0,0\r\ng00665,2,dentex_train_disease,validation,8,3,1,4,0\r\ng00666,3,dentex_train_disease,validation,2,1,1,0,0\r\ng00667,1,dentex_train_disease,validation,1,0,0,1,0\r\ng00668,1,dentex_train_disease,train,8,7,0,1,0\r\ng00670,0,dentex_train_disease,train,2,1,1,0,0\r\ng00672,2,dentex_train_disease,train,4,4,0,0,0\r\ng00673,0,dentex_train_disease,validation,7,3,1,3,0\r\ng00674,2,dentex_train_disease,train,8,2,2,4,0\r\ng00675,3,dentex_train_disease,train,2,0,1,0,1\r\ng00676,3,dentex_train_disease,train,8,6,2,0,0\r\ng00677,1,dentex_train_disease,train,12,7,1,4,0\r\ng00679,0,dentex_train_disease,train,2,2,0,0,0\r\ng00680,4,dentex_train_disease,train,3,2,1,0,0\r\ng00681,1,dentex_train_disease,train,1,0,0,1,0\r\ng00682,1,dentex_train_disease,train,2,1,1,0,0\r\ng00683,1,dentex_train_disease,train,3,3,0,0,0\r\ng00684,4,dentex_train_disease,train,9,5,1,2,1\r\ng00685,0,dentex_train_disease,train,1,1,0,0,0\r\ng00686,2,dentex_train_disease,train,7,5,0,2,0\r\ng00687,1,dentex_train_disease,train,3,3,0,0,0\r\ng00688,1,dentex_train_disease,validation,11,6,1,4,0\r\ng00692,2,dentex_train_disease,train,7,3,0,0,4\r\ng00694,3,dentex_train_disease,train,2,0,1,1,0\r\ng00695,0,dentex_train_disease,train,5,4,1,0,0\r\ng00696,4,dentex_train_disease,train,3,1,1,1,0\r\ng00697,1,dentex_train_disease,train,4,2,2,0,0\r\ng00698,3,dentex_train_disease,train,1,1,0,0,0\r\ng00700,3,dentex_train_disease,train,3,3,0,0,0\r\ng00701,2,dentex_train_disease,train,7,6,1,0,0\r\ng00702,3,dentex_train_disease,train,1,1,0,0,0\r\ng00703,4,dentex_train_disease,train,1,1,0,0,0\r\ng00704,0,dentex_train_disease,train,1,1,0,0,0\r\ng00705,0,dentex_train_disease,train,4,2,2,0,0\r\ng00706,0,dentex_train_disease,train,9,5,2,0,2\r\ng00707,2,dentex_train_disease,train,11,3,7,0,1\r\ng00708,4,dentex_train_disease,train,6,4,0,2,0\r\ng00710,2,dentex_train_disease,train,4,2,0,2,0\r\ng00711,2,dentex_train_disease,train,6,3,2,0,1\r\ng00712,2,dentex_train_disease,validation,10,5,1,4,0\r\ng00713,2,dentex_train_disease,train,8,6,0,2,0\r\ng00714,4,dentex_train_disease,validation,7,7,0,0,0\r\ng00715,3,dentex_train_disease,train,6,2,4,0,0\r\ng00717,0,dentex_train_disease,train,6,3,1,1,1\r\ng00719,2,dentex_train_disease,train,9,5,0,4,0\r\ng00720,0,dentex_train_disease,train,3,3,0,0,0\r\ng00721,1,dentex_train_disease,validation,7,2,1,4,0\r\ng00722,4,dentex_train_disease,train,6,4,0,2,0\r\ng00723,4,dentex_train_disease,train,7,5,0,2,0\r\ng00725,1,dentex_train_disease,train,9,4,1,4,0\r\ng00726,0,dentex_train_disease,train,9,3,2,4,0\r\ng00727,4,dentex_train_disease,train,2,2,0,0,0\r\ng00728,0,dentex_train_disease,train,10,6,0,4,0\r\ng00731,2,dentex_train_disease,train,7,6,0,1,0\r\ng00732,4,dentex_train_disease,train,6,6,0,0,0\r\ng00733,1,dentex_train_disease,train,8,5,1,0,2\r\ng00734,1,dentex_train_disease,validation,6,3,1,1,1\r\ng00735,2,dentex_train_disease,validation,5,2,0,3,0\r\ng00737,4,dentex_train_disease,train,3,2,0,0,1\r\ng00738,2,dentex_train_disease,train,3,2,0,1,0\r\ng00739,0,dentex_train_disease,train,7,3,2,2,0\r\ng00740,4,dentex_train_disease,validation,11,10,1,0,0\r\ng00741,0,dentex_train_disease,train,9,7,1,0,1\r\ng00742,3,dentex_train_disease,validation,4,2,0,2,0\r\ng00743,4,dentex_train_disease,train,5,3,1,0,1\r\ng00745,2,dentex_train_disease,train,1,1,0,0,0\r\ng00746,1,dentex_train_disease,train,14,7,1,3,3\r\ng00747,3,dentex_train_disease,train,3,3,0,0,0\r\ng00748,4,dentex_train_disease,train,10,2,4,4,0\r\ng00749,0,dentex_train_disease,train,9,6,0,3,0\r\ng00751,1,dentex_train_disease,train,2,2,0,0,0\r\ng00752,3,dentex_train_disease,train,4,2,2,0,0\r\ng00753,0,dentex_train_disease,validation,5,1,2,2,0\r\ng00755,4,dentex_train_disease,train,1,0,1,0,0\r\ng00756,3,dentex_train_disease,train,5,3,1,0,1\r\ng00757,2,dentex_train_disease,train,6,4,2,0,0\r\ng00758,1,dentex_train_disease,train,1,1,0,0,0\r\ng00759,1,dentex_train_disease,train,9,3,4,2,0\r\ng00760,4,dentex_train_disease,train,3,2,0,0,1\r\ng00762,4,dentex_train_disease,train,3,3,0,0,0\r\ng00766,0,dentex_train_disease,train,6,3,1,1,1\r\ng00767,2,dentex_train_disease,train,11,3,4,4,0\r\ng00768,2,dentex_train_disease,train,8,0,7,0,1\r\ng00769,4,dentex_train_disease,train,3,3,0,0,0\r\ng00770,0,dentex_train_disease,train,10,10,0,0,0\r\ng00771,1,dentex_train_disease,train,2,2,0,0,0\r\ng00773,0,dentex_train_disease,train,3,2,1,0,0\r\ng00774,1,dentex_train_disease,train,3,1,0,2,0\r\ng00775,4,dentex_train_disease,train,6,3,2,0,1\r\ng00776,0,dentex_train_disease,train,6,4,2,0,0\r\ng00779,3,dentex_train_disease,train,3,1,2,0,0\r\ng00780,0,dentex_train_disease,train,2,1,0,1,0\r\ng00781,4,dentex_train_disease,train,4,1,2,1,0\r\ng00782,4,dentex_train_disease,train,3,2,1,0,0\r\ng00783,1,dentex_train_disease,train,3,1,0,2,0\r\ng00784,1,dentex_train_disease,train,7,4,0,1,2\r\ng00785,3,dentex_train_disease,train,4,3,1,0,0\r\ng00786,4,dentex_train_disease,train,2,0,0,1,1\r\ng00787,4,dentex_train_disease,train,4,4,0,0,0\r\ng00788,2,dentex_train_disease,train,3,3,0,0,0\r\ng00789,0,dentex_train_disease,train,3,2,1,0,0\r\ng00791,4,dentex_train_disease,validation,9,5,1,2,1\r\ng00792,2,dentex_train_disease,train,4,4,0,0,0\r\ng00793,1,dentex_train_disease,validation,3,2,0,0,1\r\ng00794,1,dentex_train_disease,validation,5,5,0,0,0\r\ng00795,1,dentex_train_disease,train,6,3,0,3,0\r\ng00796,0,dentex_train_disease,train,3,2,1,0,0\r\ng00797,4,dentex_train_disease,train,6,5,1,0,0\r\ng00799,4,dentex_train_disease,train,2,0,0,2,0\r\ng00802,0,dentex_train_disease,train,2,2,0,0,0\r\ng00803,2,dentex_train_disease,train,4,0,0,4,0\r\ng00804,3,dentex_train_disease,validation,5,4,1,0,0\r\ng00805,3,dentex_train_disease,train,6,2,4,0,0\r\ng00806,3,dentex_train_disease,train,5,1,0,4,0\r\ng00807,4,dentex_train_disease,validation,8,3,3,1,1\r\ng00808,2,dentex_train_disease,train,3,0,2,1,0\r\ng00810,3,dentex_train_disease,train,11,8,0,3,0\r\ng00812,2,dentex_train_disease,validation,4,2,1,1,0\r\ng00813,1,dentex_train_disease,train,2,1,0,1,0\r\ng00816,0,dentex_train_disease,train,2,1,0,1,0\r\ng00817,0,dentex_train_disease,train,3,2,1,0,0\r\ng00818,3,dentex_train_disease,train,2,1,0,1,0\r\ng00819,3,dentex_train_disease,train,8,4,0,4,0\r\ng00820,1,dentex_train_disease,train,4,1,0,3,0\r\ng00821,1,dentex_train_disease,train,2,2,0,0,0\r\ng00822,2,dentex_train_disease,train,3,0,0,3,0\r\ng00823,2,dentex_train_disease,train,6,6,0,0,0\r\ng00824,1,dentex_train_disease,validation,6,4,2,0,0\r\ng00825,1,dentex_train_disease,validation,5,5,0,0,0\r\ng00826,1,dentex_train_disease,train,10,4,5,0,1\r\ng00827,1,dentex_train_disease,train,5,1,0,4,0\r\ng00828,3,dentex_train_disease,train,6,4,1,0,1\r\ng00832,0,dentex_train_disease,train,6,3,3,0,0\r\ng00833,3,dentex_train_disease,train,1,1,0,0,0\r\ng00834,3,dentex_train_disease,train,5,2,3,0,0\r\ng00835,4,dentex_train_disease,train,10,7,3,0,0\r\ng00837,0,dentex_train_disease,train,4,3,0,1,0\r\ng00838,2,dentex_train_disease,train,1,1,0,0,0\r\ng00839,4,dentex_train_disease,train,1,0,1,0,0\r\ng00840,3,dentex_train_disease,train,7,2,2,3,0\r\ng00841,3,dentex_train_disease,train,5,4,1,0,0\r\ng00842,2,dentex_train_disease,train,4,3,1,0,0\r\ng00843,3,dentex_train_disease,train,7,7,0,0,0\r\ng00844,3,dentex_train_disease,train,11,5,3,3,0\r\ng00845,3,dentex_train_disease,train,2,2,0,0,0\r\ng00847,1,dentex_train_disease,train,11,10,1,0,0\r\ng00848,0,dentex_train_disease,train,1,1,0,0,0\r\ng00849,2,dentex_train_disease,train,3,1,1,0,1\r\ng00850,0,dentex_train_disease,train,3,1,0,2,0\r\ng00851,1,dentex_train_disease,validation,2,1,0,1,0\r\ng00852,2,dentex_train_disease,train,1,1,0,0,0\r\ng00853,0,dentex_train_disease,train,15,12,2,0,1\r\ng00854,0,dentex_train_disease,train,3,3,0,0,0\r\ng00855,2,dentex_train_disease,train,6,3,3,0,0\r\ng00856,3,dentex_train_disease,validation,4,2,0,2,0\r\ng00857,1,dentex_train_disease,train,7,6,1,0,0\r\ng00859,0,dentex_train_disease,train,16,7,4,2,3\r\ng00860,1,dentex_train_disease,train,14,7,6,1,0\r\ng00861,0,dentex_train_disease,train,7,4,1,2,0\r\ng00862,4,dentex_train_disease,train,5,2,3,0,0\r\ng00863,4,dentex_train_disease,train,13,8,0,2,3\r\ng00864,4,dentex_train_disease,train,5,2,1,2,0\r\ng00865,1,dentex_train_disease,train,2,2,0,0,0\r\ng00866,4,dentex_train_disease,train,2,2,0,0,0\r\ng00867,4,dentex_train_disease,train,1,1,0,0,0\r\ng00868,4,dentex_train_disease,train,9,7,1,0,1\r\ng00869,3,dentex_train_disease,train,2,1,1,0,0\r\ng00870,2,dentex_train_disease,train,5,4,1,0,0\r\ng00871,1,dentex_train_disease,train,7,7,0,0,0\r\ng00872,1,dentex_train_disease,train,4,1,3,0,0\r\ng00874,3,dentex_train_disease,train,3,3,0,0,0\r\ng00875,4,dentex_train_disease,train,5,3,2,0,0\r\ng00876,4,dentex_train_disease,train,7,2,1,1,3\r\ng00877,3,dentex_train_disease,train,3,2,1,0,0\r\ng00878,4,dentex_train_disease,train,7,5,0,2,0\r\ng00879,3,dentex_train_disease,train,4,0,2,1,1\r\ng00880,1,dentex_train_disease,train,4,0,0,4,0\r\ng00881,2,dentex_train_disease,train,2,2,0,0,0\r\ng00882,2,dentex_train_disease,train,8,3,1,3,1\r\ng00883,4,dentex_train_disease,train,3,2,1,0,0\r\ng00884,2,dentex_train_disease,train,11,4,0,4,3\r\ng00885,1,dentex_train_disease,train,9,3,2,4,0\r\ng00886,4,dentex_train_disease,train,10,6,3,0,1\r\ng00887,2,dentex_train_disease,train,4,4,0,0,0\r\ng00888,1,dentex_train_disease,train,6,6,0,0,0\r\ng00889,3,dentex_train_disease,train,1,1,0,0,0\r\ng00891,0,dentex_train_disease,train,1,1,0,0,0\r\ng00894,1,dentex_train_disease,train,2,2,0,0,0\r\ng00895,3,dentex_train_disease,train,3,0,0,3,0\r\ng00896,3,dentex_train_disease,train,10,8,1,1,0\r\ng00897,0,dentex_train_disease,train,4,1,0,3,0\r\ng00898,1,dentex_train_disease,train,2,1,1,0,0\r\ng00899,1,dentex_train_disease,train,3,3,0,0,0\r\ng00902,3,dentex_train_disease,train,7,4,1,0,2\r\ng00903,2,dentex_train_disease,train,2,0,0,2,0\r\ng00904,1,dentex_train_disease,train,8,4,0,4,0\r\ng00905,3,dentex_train_disease,train,6,2,1,3,0\r\ng00906,2,dentex_train_disease,train,12,9,3,0,0\r\ng00907,4,dentex_train_disease,train,2,2,0,0,0\r\ng00909,3,dentex_train_disease,train,7,4,3,0,0\r\ng00910,4,dentex_train_disease,train,10,9,1,0,0\r\ng00912,1,dentex_train_disease,train,4,2,0,2,0\r\ng00913,2,dentex_train_disease,train,2,2,0,0,0\r\ng00914,3,dentex_train_disease,train,3,2,1,0,0\r\ng00915,4,dentex_train_disease,train,4,3,1,0,0\r\ng00916,1,dentex_train_disease,train,5,2,3,0,0\r\ng00917,3,dentex_train_disease,train,6,2,0,4,0\r\ng00918,4,dentex_train_disease,train,3,1,2,0,0\r\ng00920,3,dentex_train_disease,train,6,2,0,4,0\r\ng00921,1,dentex_train_disease,train,6,5,0,0,1\r\ng00922,4,dentex_train_disease,train,14,10,2,0,2\r\ng00923,4,dentex_train_disease,train,3,2,0,1,0\r\ng00925,0,dentex_train_disease,train,9,6,1,1,1\r\ng00926,2,dentex_train_disease,validation,4,4,0,0,0\r\ng01008,4,dentex_official_val,train,3,3,0,0,0\r\ng01061,2,dentex_official_val,train,6,5,1,0,0\r\ng01071,2,dentex_official_val,train,2,0,1,1,0\r\ng01083,0,dentex_official_val,train,2,2,0,0,0\r\ng01093,1,dentex_official_val,train,6,3,1,2,0\r\ng01138,2,dentex_official_val,train,2,2,0,0,0\r\ng01156,3,dentex_official_val,train,5,2,1,1,1\r\ng01174,2,dentex_official_val,train,4,4,0,0,0\r\ng01190,0,dentex_official_val,train,8,7,0,1,0\r\ng01216,3,dentex_official_val,train,4,4,0,0,0\r\ng01263,1,dentex_official_val,train,2,2,0,0,0\r\ng01280,0,dentex_official_val,train,2,2,0,0,0\r\ng01288,4,dentex_official_val,train,3,0,2,1,0\r\ng01289,4,dentex_official_val,train,7,4,3,0,0\r\ng01290,1,dentex_official_val,train,3,0,3,0,0\r\ng01291,4,dentex_official_val,train,3,2,0,1,0\r\ng01292,2,dentex_official_val,train,7,3,0,4,0\r\ng01293,3,dentex_official_val,train,6,2,4,0,0\r\ng01295,2,dentex_official_val,train,3,1,1,0,1\r\ng01296,3,dentex_official_val,train,2,0,1,0,1\r\ng01297,0,dentex_official_val,train,1,1,0,0,0\r\ng01299,3,dentex_official_val,train,2,2,0,0,0\r\ng01300,0,dentex_official_val,train,8,4,0,4,0\r\ng01301,0,dentex_official_val,train,11,4,1,4,2\r\ng01302,0,dentex_official_val,train,1,0,1,0,0\r\ng01303,4,dentex_official_val,train,5,1,2,0,2\r\ng01304,3,dentex_official_val,train,9,5,0,4,0\r\ng01305,2,dentex_official_val,train,4,2,2,0,0\r\ng01306,1,dentex_official_val,train,2,1,0,0,1\r\ng01307,4,dentex_official_val,train,3,3,0,0,0\r\ng01308,1,dentex_official_val,train,5,0,0,4,1\r\ng01309,1,dentex_official_val,train,1,0,1,0,0\r\ng01310,0,dentex_official_val,train,6,2,2,2,0\r\ng01311,3,dentex_official_val,train,4,3,1,0,0\r\ng01312,4,dentex_official_val,train,4,3,1,0,0\r\ng01313,1,dentex_official_val,train,2,2,0,0,0\r\ng01314,1,dentex_official_val,train,2,2,0,0,0\r\ng01315,1,dentex_official_val,train,4,1,0,3,0\r\ng01316,4,dentex_official_val,train,3,1,0,2,0\r\ng01317,1,dentex_official_val,train,2,0,0,2,0\r\n"}
EMBEDDED_SHA256 = {"kaggle/cv5_lib.py": "72084f9b7067f71f808880fb70b58e546405f3d970f0e30be52dd93bdaea6b40", "scripts/dataset/make_cv_folds.py": "283fa080ed97c542b942611caa5d7c64230b7b06f15a7ea7ad652c98279ecff6", "data/splits/cv5_folds_v1.csv": "9b767a4822844ccf0c4bf33d299e6e37420d1458a38035af0c3a3cb9107b47a6"}
for rel, txt in EMBEDDED.items():
    assert hashlib.sha256(txt.encode("utf-8")).hexdigest() == EMBEDDED_SHA256[rel], f"embedded file changed: {rel}"
    (CODE / rel).parent.mkdir(parents=True, exist_ok=True)
    (CODE / rel).write_text(txt, encoding="utf-8", newline="")
    log("embedded", rel, EMBEDDED_SHA256[rel][:16])
import cv5_lib as CV
assert CV.sha256_file(CODE / "data/splits/cv5_folds_v1.csv") == CV.FOLD_FILE_SHA256

In [ ]:
# ---- 3. folds: approved file + gate 1 (protocol sections 2 and 7) -------------------------------------------------------
SPLITS = CODE / "data/splits"
ROWS = {s: DK.load_split(SPLITS, s) for s in ("train", "validation", "calibration", "test")}   # calibration/test: ids only
FOLD_OF = CV.load_folds(SPLITS / "cv5_folds_v1.csv")
GATE1 = CV.check_folds(FOLD_OF, ROWS)
log("gate 1 passed:", json.dumps({k: GATE1[k] for k in ("pool_groups", "pool_findings", "radiographs_per_fold")}))
POOL = [r for s in CV.POOL_SPLITS for r in ROWS[s]]
PRIMARY_ROWS = [r for r in POOL if r["source_dataset"] == CV.PRIMARY]
SECONDARY_ROWS = [r for r in POOL if r["source_dataset"] == CV.SECONDARY]
assert len(PRIMARY_ROWS) == 2851 and len(SECONDARY_ROWS) == 159, (len(PRIMARY_ROWS), len(SECONDARY_ROWS))
del ROWS["calibration"], ROWS["test"]          # nothing below can use them
PLANS = {k: CV.fold_plan(POOL, FOLD_OF, k) for k in FOLDS}
GATE2 = {k: CV.check_plan(PLANS[k], FOLD_OF) for k in FOLDS}
for k, g in GATE2.items():
    log(f"gate 2 fold {k}: fit {g['fit']['radiographs']} r / {g['fit']['findings']} f, inner {g['inner']['radiographs']} r / "
        f"{g['inner']['findings']} f, held-out {g['heldout']['radiographs']} r / {g['heldout']['findings']} f "
        f"(+ official-val {g['heldout_secondary']['findings']} f)")
(ART / "fold_plan.json").write_text(json.dumps({"fold_file_sha256": CV.FOLD_FILE_SHA256, "gate1": GATE1, "gate2": GATE2,
    "groups": {k: {n: sorted({r["radiograph_group"] for r in PLANS[k][n]}) for n in ("fit", "inner", "heldout", "heldout_secondary")}
               for k in FOLDS}}, indent=1))

In [ ]:
# ---- 4. images of the pool only (official archive, SHA-256 verified per image) ---------------------------------------------
dis = "training_data/quadrant-enumeration-disease"
if DK.verify_images([r for r in POOL if r["image_path"].startswith(dis)], DATA_ROOT)[1]:
    log("fetching the DENTEX disease subset from the official archive (about 2.9 GB, range requests)")
    t = time.time()
    DX.fetch_dentex_disease_subset(DATA_ROOT)
    log(f"  fetched in {time.time() - t:.0f}s")
vd = [r for r in POOL if r["image_path"].startswith("validation_data/")]
if vd and DK.verify_images(vd, DATA_ROOT)[1]:
    spec = DX.DOWNLOADS["dentex_hf_validation_images"]
    zp = DATA_ROOT / "_archives/validation_data.zip"
    log("official validation images:", DX.download(spec["url"], zp, spec["bytes"], spec["sha256"]))
    DX.safe_extract(zp, DATA_ROOT)
n_ok, problems = DK.verify_images(POOL, DATA_ROOT)
log(f"pool images verified: {n_ok} radiographs, problems: {problems}")
assert not problems, "stop: pool images missing or not byte-identical to the manifests"

In [ ]:
# ---- 5. crops (production crop rule) for every pool finding ---------------------------------------------------------------
t = time.time()
X_ALL, Y_ALL, _ = DK.build_crops(POOL, DATA_ROOT, CACHE / "pool_gt.npz")
IDX = {r["sample_id"]: i for i, r in enumerate(POOL)}
log(f"GT crops: {len(Y_ALL)} findings [{time.time() - t:.0f}s]")

In [ ]:
# ---- 6. production-detector boxes for every pool radiograph + gate 3 ------------------------------------------------------
DET_CACHE = ART / "detector_boxes_pool.json"
det_boxes = json.loads(DET_CACHE.read_text()) if DET_CACHE.is_file() else {}
det = CV.ProductionDetector(CODE / "data/checkpoints/dentex_teeth_detection_v1.pth")
t = time.time()
for r in POOL:
    k = r["project_image_id"]
    if k not in det_boxes:
        det_boxes[k] = det.boxes(DATA_ROOT / r["image_path"])
DET_CACHE.write_text(json.dumps(det_boxes))
del det
log(f"detector boxes for {len(det_boxes)} radiographs [{time.time() - t:.0f}s]")
cached = json.loads((CODE / "docs/audit_evidence/ml_improvement/crop_robustness/val_detector_boxes.json").read_text())
GATE3 = CV.compare_detector(ROWS["validation"], det_boxes, cached)
log("gate 3 (detector reproduction on the 65 old-validation radiographs):",
    {k: v for k, v in GATE3.items() if k != "images_with_different_match"})
DET_OK = GATE3["pass"]
if not DET_OK:
    log("GATE 3 FAILED: detector-box results will NOT be reported (protocol section 7); GT-box results still are")
XD, MI, MISSED = DK.build_detector_crops(POOL, DATA_ROOT, DET_CACHE, lambda r: r["project_image_id"])
MATCHED = np.zeros(len(POOL), bool)
MATCHED[MI] = True
XD_OF = np.full(len(POOL), -1)
XD_OF[MI] = np.arange(len(MI))
log(f"detector-matched findings: {int(MATCHED.sum())} / {len(POOL)} (unmatched {MISSED})")

In [ ]:
# ---- 7. train 5 folds x seeds (resumable, isolated processes) and predict each held-out fold once ----------------------------
import torch, torchvision as tv
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAILED = []

def budget_left():
    return TIME_BUDGET_H * 3600 - (time.time() - T0)

def predict_heldout(ck, rows):
    if not rows:
        return np.zeros(0, int), np.zeros((0, len(DK.CLASSES))), np.zeros((0, len(DK.CLASSES)))
    m = DK.new_model(torch, tv, pretrained=False)
    m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=True), strict=True)
    m.to(dev)
    ii = np.array([IDX[r["sample_id"]] for r in rows])
    pg = DK.predict(torch, m, X_ALL[ii], dev)
    pd = np.full_like(pg, np.nan)
    has = MATCHED[ii]
    if has.any():
        pd[has] = DK.predict(torch, m, XD[XD_OF[ii[has]]], dev)
    del m
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return ii, pg, pd

for k in FOLDS:
    plan = PLANS[k]
    take = lambda rows: {"X": X_ALL[[IDX[r["sample_id"]] for r in rows]], "y": Y_ALL[[IDX[r["sample_id"]] for r in rows]]}
    fit_npz = DK.save_arrays(CACHE / f"f{k}_fit.npz", take(plan["fit"]))
    inner_npz = DK.save_arrays(CACHE / f"f{k}_inner.npz", take(plan["inner"]))
    cfg = CV.fold_config(k)
    for sd in SEEDS:
        out = RUNS / f"{cfg.name}_s{sd}"
        if (out / "heldout_pred.npz").is_file():
            log(f"{cfg.name} s{sd}: already predicted - skipped")
            continue
        if budget_left() < 900:
            log("TIME BUDGET reached: no new run started. Save the output, then rerun with RESUME_FROM set.")
            break
        r = DK.train_run_isolated(cfg, sd, fit_npz, inner_npz, RUNS, log=log)
        if r.get("failed"):
            FAILED.append(f"{cfg.name}_s{sd}")
            log(f"{cfg.name} s{sd}: FAILED after retries - see {out}/failure.json")
            continue
        ii, pg, pd = predict_heldout(out / "best.pth", plan["heldout"])
        ii2, pg2, pd2 = predict_heldout(out / "best.pth", plan["heldout_secondary"])
        np.savez(out / "heldout_pred.npz", idx=ii, gt=pg, det=pd, idx_sec=ii2, gt_sec=pg2, det_sec=pd2)
        (out / "cv_run.json").write_text(json.dumps({"fold": k, "seed": sd, "fit_findings": len(plan["fit"]),
            "inner_findings": len(plan["inner"]), "heldout_findings": len(plan["heldout"]),
            "heldout_secondary_findings": len(plan["heldout_secondary"]), "best_epoch": r["best_epoch"],
            "inner_val_macro_f1": r["val_gt"]["macro_f1"], "checkpoint_sha256": r["checkpoint_sha256"],
            "note": "result.json split_note is the generic BASE text; for CV the validation set is the fold's inner validation subset"}, indent=1))
        log(f"{cfg.name} s{sd}: best ep {r['best_epoch']} inner-val macro-F1 {r['val_gt']['macro_f1']:.4f}  "
            f"elapsed {(time.time() - T0) / 3600:.2f} h")
log("failed runs:", FAILED or "none")

In [ ]:
# ---- 8. out-of-fold evaluation (gates 4 and 5), aggregation, bootstrap, Caries vs Deep Caries ----------------------------------
P_IDX = np.array([IDX[r["sample_id"]] for r in PRIMARY_ROWS])
S_IDX = np.array([IDX[r["sample_id"]] for r in SECONDARY_ROWS])
pos = {g: i for i, g in enumerate(P_IDX)}
spos = {g: i for i, g in enumerate(S_IDX)}
n, ns, C = len(P_IDX), len(S_IDX), len(DK.CLASSES)
OOF, OOF_S, FOLD_OF_FINDING = {}, {}, np.full(n, -1)
missing_runs = []
for sd in SEEDS:
    o = {"gt": np.full((n, C), np.nan), "det": np.full((n, C), np.nan), "filled": np.zeros(n, int)}
    os_ = {"gt": np.full((ns, C), np.nan), "det": np.full((ns, C), np.nan)}
    for k in FOLDS:
        f = RUNS / f"{CV.run_name(k)}_s{sd}" / "heldout_pred.npz"
        if not f.is_file():
            missing_runs.append(f.parent.name)
            continue
        z = np.load(f)
        rows_i = np.array([pos[i] for i in z["idx"]])
        o["gt"][rows_i], o["det"][rows_i] = z["gt"], z["det"]
        o["filled"][rows_i] += 1
        FOLD_OF_FINDING[rows_i] = k
        if len(z["idx_sec"]):
            si = np.array([spos[i] for i in z["idx_sec"]])
            os_["gt"][si], os_["det"][si] = z["gt_sec"], z["det_sec"]
    OOF[sd], OOF_S[sd] = o, os_
assert not FAILED and not missing_runs, f"gate 4: runs missing {FAILED + missing_runs} - resume with RESUME_FROM; nothing is reported"
CV.check_oof(OOF, n, SEEDS)
log("gates 4 and 5 passed: every primary finding has exactly one out-of-fold prediction per seed")

y = Y_ALL[P_IDX]
m = MATCHED[P_IDX]
groups = np.array([r["radiograph_group"] for r in PRIMARY_ROWS])
POOLED = CV.pooled_reports(y, OOF, m, SEEDS)
settings = ["gt"] + (["det_matched"] if DET_OK else [])
SUMMARY = {s: {"macro_f1": CV.mean_sd([POOLED[sd][s]["macro_f1"] for sd in SEEDS]),
               "accuracy": CV.mean_sd([POOLED[sd][s]["accuracy"] for sd in SEEDS]),
               "ece_10bin": CV.mean_sd([POOLED[sd][s]["ece_10bin"] for sd in SEEDS]),
               "per_class": {c: {mm: CV.mean_sd([POOLED[sd][s]["per_class"][c][mm] for sd in SEEDS]) for mm in ("precision", "recall", "f1")}
                             for c in DK.CLASSES},
               "confusion_sum_over_seeds": np.sum([POOLED[sd][s]["confusion"] for sd in SEEDS], axis=0).tolist()} for s in settings}
BOOT = {"gt": CV.cluster_bootstrap(y, [OOF[sd]["gt"].argmax(1) for sd in SEEDS], groups)}
if DET_OK:
    BOOT["det_matched"] = CV.cluster_bootstrap(y, [np.nan_to_num(OOF[sd]["det"], nan=0).argmax(1) for sd in SEEDS], groups, valid=m)
PER_FOLD = {}
for k in FOLDS:
    sub = np.flatnonzero(FOLD_OF_FINDING == k)
    rep = CV.pooled_reports(y[sub], {sd: {kk: OOF[sd][kk][sub] for kk in ("gt", "det")} for sd in SEEDS}, m[sub], SEEDS)
    PER_FOLD[k] = {"radiographs": int(len(set(groups[sub]))), "findings": int(len(sub)), "detector_matched": int(m[sub].sum()),
                   **{s: {"macro_f1": CV.mean_sd([rep[sd][s]["macro_f1"] for sd in SEEDS]),
                          "per_class": {c: {mm: CV.mean_sd([rep[sd][s]["per_class"][c][mm] for sd in SEEDS]) for mm in ("precision", "recall", "f1")}
                                        for c in DK.CLASSES},
                          "confusion_sum_over_seeds": np.sum([rep[sd][s]["confusion"] for sd in SEEDS], axis=0).tolist()} for s in settings}}
FOLD_LEVEL = {s: CV.mean_sd([PER_FOLD[k][s]["macro_f1"]["mean"] for k in FOLDS]) for s in settings}
old_val = np.array([r["split"] == "validation" for r in PRIMARY_ROWS])
UNSEEN = CV.pooled_reports(y[old_val], {sd: {kk: OOF[sd][kk][old_val] for kk in ("gt", "det")} for sd in SEEDS}, m[old_val], SEEDS)
DET_UNSEEN = {"population": "65 old-validation radiographs (not in the detector's training assignment)", "findings": int(old_val.sum()),
              "macro_f1_det_matched": CV.mean_sd([UNSEEN[sd]["det_matched"]["macro_f1"] for sd in SEEDS]) if DET_OK else None,
              "macro_f1_gt": CV.mean_sd([UNSEEN[sd]["gt"]["macro_f1"] for sd in SEEDS])}
ys = Y_ALL[S_IDX]
SECONDARY = {"population": "DENTEX official-validation findings (never trained on)", "findings": int(ns),
             "gt_macro_f1": CV.mean_sd([DK.report(ys, OOF_S[sd]["gt"].argmax(1))["macro_f1"] for sd in SEEDS])}
CARIES = CV.caries_confusion(PRIMARY_ROWS, OOF, m, SEEDS)

for s in settings:
    log(f"pooled out-of-fold macro-F1 ({s}): {SUMMARY[s]['macro_f1']['mean']:.4f} +- {SUMMARY[s]['macro_f1']['sd']:.4f} over seeds; "
        f"radiograph bootstrap 95% CI {BOOT[s]['ci95'][0]:.4f}-{BOOT[s]['ci95'][1]:.4f}; fold-level mean {FOLD_LEVEL[s]['mean']:.4f} "
        f"+- {FOLD_LEVEL[s]['sd']:.4f}")
log("Caries/Deep Caries findings confused by all seeds (GT boxes):", CARIES["confused_by_all_seeds_gt"], "of", CARIES["n"])

RESULTS = {"protocol": "reports/cv5_protocol.md", "smoke": SMOKE, "seeds": SEEDS, "folds": FOLDS, "environment": env,
           "gates": {"1_folds": True, "2_plans": True, "3_detector": {k: v for k, v in GATE3.items()}, "4_runs_complete": True,
                     "5_one_prediction_per_finding": True},
           "population": {"primary_findings": int(n), "primary_radiographs": int(len(set(groups))), "detector_matched": int(m.sum())},
           "pooled_per_seed": {str(sd): POOLED[sd] for sd in SEEDS}, "summary": SUMMARY, "bootstrap": BOOT,
           "per_fold": {str(k): v for k, v in PER_FOLD.items()}, "fold_level_macro_f1": FOLD_LEVEL,
           "detector_unseen_subset": DET_UNSEEN, "secondary_official_val": SECONDARY,
           "caries_confusion": {k: v for k, v in CARIES.items() if k != "records"}}
(ART / "cv5_results.json").write_text(json.dumps(RESULTS, indent=1, default=float))
import csv
with open(ART / "caries_confusion_findings.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(CARIES["records"][0]))
    w.writeheader()
    w.writerows(CARIES["records"])
with open(ART / "oof_predictions.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["sample_id", "radiograph_group", "fold", "true_class", "detector_matched"] +
               [f"s{sd}_{s}_{c}" for sd in SEEDS for s in ("gt", "det") for c in DK.CLASSES])
    for i, r in enumerate(PRIMARY_ROWS):
        w.writerow([r["sample_id"], r["radiograph_group"], int(FOLD_OF_FINDING[i]), r["target_class"], bool(m[i])] +
                   [("" if np.isnan(v) else round(float(v), 6)) for sd in SEEDS for s in ("gt", "det") for v in OOF[sd][s][i]])

In [ ]:
# ---- 9. package artifacts ---------------------------------------------------------------------------------------------------
pkg = ART / "package"
if pkg.exists():
    shutil.rmtree(pkg)
(pkg / "runs").mkdir(parents=True)
ck = {}
for d in sorted(RUNS.iterdir()):
    for f in ("result.json", "cv_run.json"):
        if (d / f).is_file():
            shutil.copy2(d / f, pkg / "runs" / f"{d.name}__{f}")
    if (d / "heldout_pred.npz").is_file():
        shutil.copy2(d / "heldout_pred.npz", pkg / "runs" / f"{d.name}__heldout_pred.npz")
    if (d / "best.pth").is_file():
        ck[d.name] = {"sha256": CV.sha256_file(d / "best.pth"), "bytes": (d / "best.pth").stat().st_size,
                      "kept_at": f"notebook output cv5/runs/{d.name}/best.pth"}
(pkg / "checkpoint_manifest.json").write_text(json.dumps(ck, indent=1))
for f in ("cv5_results.json", "oof_predictions.csv", "caries_confusion_findings.csv", "fold_plan.json", "detector_boxes_pool.json",
          "training_log.txt", "environment.json"):
    if (ART / f).is_file():
        shutil.copy2(ART / f, pkg / f)
for rel in ("data/splits/cv5_folds_v1.csv", "kaggle/cv5_lib.py", "scripts/dataset/make_cv_folds.py"):
    (pkg / rel).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CODE / rel, pkg / rel)
(pkg / "experiment_configs.json").write_text(json.dumps({CV.run_name(k): dataclasses.asdict(CV.fold_config(k)) for k in FOLDS}, indent=1))
(pkg / "README.md").write_text("\n".join([
    "# DentaMind CV5-BASE artifacts" + (" (SMOKE - pipeline check, NOT a result)" if SMOKE else ""), "",
    f"Protocol reports/cv5_protocol.md. Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} on {env.get('gpu')} "
    f"(torch {env['torch']}), runtime {(time.time() - T0) / 3600:.2f} h. Seeds {SEEDS}, folds {FOLDS}.", "",
    "No test or calibration image was read. No model was promoted. Checkpoints stay in the notebook output (hashes in",
    "checkpoint_manifest.json); per-run held-out predictions are in runs/*__heldout_pred.npz.", "",
    "Main numbers: cv5_results.json (summary, bootstrap, per_fold, detector_unseen_subset, secondary_official_val, caries_confusion)."]))
sums = [f"{CV.sha256_file(p)}  {p.relative_to(pkg).as_posix()}" for p in sorted(pkg.rglob("*")) if p.is_file()]
(pkg / "SHA256SUMS.txt").write_text("\n".join(sums) + "\n")
zp = WORK / ("dentamind_5fold_cv_artifacts_SMOKE.zip" if SMOKE else "dentamind_5fold_cv_artifacts.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(pkg.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(pkg).as_posix())
log(f"ARCHIVE {zp}  {zp.stat().st_size / 2**20:.1f} MB  sha256 {CV.sha256_file(zp)}")